# 📄 Gerando Dataset de Instruções para Fine‑Tuning a partir de Documentos

Este tutorial ensina como transformar conhecimento bruto — contido em manuais, guias ou artigos (PDF/Texto) — em um conjunto de dados no formato **instruction‑input‑output**, pronto para ser usado em fine‑tuning de modelos de linguagem (como o que fizemos com LoRA).

**Por que isso é importante?**
- Modelos pré‑treinados possuem grande capacidade, mas frequentemente carecem de conhecimento específico de domínio.
- Ao criar pares pergunta‑resposta baseados em documentos reais, podemos ensinar o modelo a responder corretamente sobre aquele domínio.
- O formato JSONL utilizado é o padrão para *instruction tuning*, amplamente adotado em projetos como Alpaca, Dolly, etc.

**O que você aprenderá:**
- Extrair texto de arquivos PDF.
- Dividir o texto em trechos (chunks) adequados para processamento.
- Usar tanto um modelo **seq2seq** (encoder‑decoder) quanto um modelo **causal** (autoregressivo) para gerar automaticamente triplas `instruction`, `input` (opcional) e `output`.
- Estruturar e salvar os dados em JSONL.
- Comparar a qualidade dos dados gerados com uma extração manual simples.

## 📚 1. Fundamentos da Geração de Dados Instrucionais

O objetivo é, dado um trecho de texto $D$ (ex.: um parágrafo de manual), produzir uma tripla $(I, X, O)$ onde:
- $I$: instrução (pergunta ou comando)
- $X$: entrada adicional (opcional, pode ser vazia)
- $O$: saída desejada, baseada no conteúdo de $D$

Formalmente, queremos modelar uma distribuição condicional:

$$P(I, X, O \mid D)$$

Utilizamos um LLM para amostrar dessas distribuições, fornecendo um *prompt* que instrui o modelo a gerar o JSON desejado a partir do contexto.

**Por que isso funciona?**  
Modelos de linguagem modernos, quando condicionados com prompts adequados, conseguem realizar *in‑context learning* – eles entendem a tarefa descrita e produzem texto estruturado. A qualidade depende fortemente:
- Da capacidade do modelo (ex.: 7B+ parâmetros).
- Da clareza do prompt.
- Da temperatura (controla a criatividade).

Neste notebook demonstraremos dois paradigmas:
- **Seq2Seq** (ex.: FLAN‑T5): recebe o texto inteiro e gera a saída condicionada ao *encoder*.
- **Causal** (ex.: GPT‑2): gera a continuação de um prompt, simulando a tarefa como um "completamento" de texto.

> **Nota didática:** Utilizaremos modelos pequenos para demonstrar o fluxo. Em aplicações reais, recomenda‑se modelos maiores (LLaMA 3, GPT‑4, etc.) para melhores resultados.

## 📦 2. Requisitos

Instale as bibliotecas necessárias (descomente a linha se ainda não as tiver):

In [ ]:
!pip install transformers datasets pypdf2 accelerate sentencepiece

In [2]:
!pip install --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 1.9 MB/s  0:00:05m0:00:0100:01
  Attempting uninstall: transformers
    Found existing installation: transformers 5.10.2
    Uninstalling transformers-5.10.2:
      Successfully uninstalled transformers-5.10.2


In [3]:
import json
import re
from pathlib import Path
import torch

from pypdf import PdfReader
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM

/home/henderson/Documentos/Rag/Pipeline-RAG-com-Fine-Tuning-LoRA-e-Disponibiliza-o-via-API-RESTful/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Modo de execução: quando True, carregamos/aplicamos apenas o modelo TunAI
USE_ONLY_TUNAI = True


## 📥 3. Carregar e Extrair Texto de um PDF

Vamos usar um arquivo PDF de exemplo: `manual.pdf`. O código abaixo extrai todo o texto de todas as páginas.

In [5]:
def extract_text_from_pdf(pdf_path):
    """Extrai texto de um arquivo PDF."""
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"
    return text

# Substitua pelo caminho do seu PDF
pdf_path = "RelatGestSaud.pdf"
full_text = extract_text_from_pdf(pdf_path)
print(f"Total de caracteres extraídos: {len(full_text)}")
print("\n--- INÍCIO DO TEXTO ---\n")
print(full_text[:500])

Total de caracteres extraídos: 648768

--- INÍCIO DO TEXTO ---

RELATÓRIO DE GESTÃO

MENSAGEM DO MINISTRO 2
VISÃO GERAL ORGANIZACIONAL 
E GOVERNANÇA 3
1.1 Identificação da UPC  
 (Unidade Prestadora de Contas) 4
1.2   Estrutura Organizacional 5
1.3   Cadeia de Valor 9
1.4  Mapa Estratégico 10 
1.5   Políticas Estratégicas 11 
1.6   Planejamento e Monitoramento 13 
1.7   Descrição dos objetivos do Exercício 17 
1.8   Monitoramento dos Instrumentos  
              de Planejamento 19
1.9   Estrutura de Governança  23
1.10 Oportunidades e Perspectivas 26 
CONFOR


## ✂️ 4. Dividir o Texto em *Chunks* (Pedaços)

Modelos de linguagem têm um limite máximo de tokens de entrada (janela de contexto). Precisamos quebrar o texto em segmentos que caibam nessa janela, mas mantendo significado.  
Estratégias comuns:
- Divisão por parágrafos.
- Divisão com sobreposição (*sliding window*) para evitar perda de contexto nas bordas.

Aqui faremos uma divisão simples: a cada `max_chunk_chars` caracteres, garantindo que a quebra ocorra em um espaço (para não cortar palavras).

In [6]:
def chunk_text(text, max_chunk_chars=400):  #400 e 800
    """Divide o texto em blocos de no máximo max_chunk_chars caracteres."""
    words = text.split()
    chunks = []
    current_chunk = ""
    for word in words:
        if len(current_chunk) + len(word) + 1 <= max_chunk_chars:
            current_chunk += (" " if current_chunk else "") + word
        else:
            chunks.append(current_chunk)
            current_chunk = word
    if current_chunk:
        chunks.append(current_chunk)
    return chunks

chunks = chunk_text(full_text, max_chunk_chars=200)  # pequeno para demonstração
print(f"Número de chunks: {len(chunks)}")
print(f"Exemplo de chunk (primeiro):\n{chunks[0][:300]}...")

Número de chunks: 3234
Exemplo de chunk (primeiro):
RELATÓRIO DE GESTÃO MENSAGEM DO MINISTRO 2 VISÃO GERAL ORGANIZACIONAL E GOVERNANÇA 3 1.1 Identificação da UPC (Unidade Prestadora de Contas) 4 1.2 Estrutura Organizacional 5 1.3 Cadeia de Valor 9 1.4...


## 🎯 5. Projetar o *Prompt* para Geração

O prompt deve instruir o modelo a produzir um JSON válido com os campos `instruction`, `input` e `output`. Como trabalharemos com duas arquiteturas, precisamos de prompts ligeiramente diferentes:

- **Seq2Seq**: o prompt é enviado diretamente como entrada, e o modelo gera a saída condicionada. Exemplo:
```
You are an AI assistant... 
Text: {chunk}
JSON: 
```
- **Causal**: o modelo vê o prompt como um prefixo e deve continuar a sequência. Precisamos estruturar o prompt de forma que o modelo "complete" com o JSON. Exemplo (estilo Alpaca):
```
### Instruction: ... 
### Input: {chunk}
### Response: 
```

Utilizaremos `temperature=0.3` para equilibrar criatividade e fidelidade.

## 🤖 6. Carregar Modelos de Linguagem: Seq2Seq e Causal

Carregaremos quatro modelos entre 1.5B e 4B:
- `t5-3b` (seq2seq) – gerado diretamente com `model.generate()`
- `google/flan-t5-xl` (seq2seq) – gerado diretamente com `model.generate()`
- `EleutherAI/gpt-neo-2.7B` (causal) – pipeline `text-generation`
- `facebook/opt-2.7b` (causal) – pipeline `text-generation`

> **Importante:** nesta versão do `transformers`, os tasks `summarization` e `text2text-generation` não estão disponíveis para os modelos seq2seq, então usamos a geração direta nestes casos.

> **Dica:** use quantização e `device_map="auto"` se tiver GPU com memória limitada.


In [15]:
causal_id = "unsloth/Llama-3.2-3B-Instruct"

# Carrega o tokenizador padrão
causal_tokenizer = AutoTokenizer.from_pretrained(causal_id)
causal_tokenizer.pad_token = causal_tokenizer.eos_token 

# Carrega o modelo de forma otimizada para sua GPU
causal_model = AutoModelForCausalLM.from_pretrained(
    causal_id,
    torch_dtype=torch.float16,  # Roda super rápido consumindo metade da memória
    device_map="auto"           # Joga o modelo automaticamente na GPU
)

# Cria o pipeline de geração de texto
TunAI_causal = pipeline(
    "text-generation",
    model=causal_model,
    tokenizer=causal_tokenizer,
    max_new_tokens=256,
    do_sample=True,
    temperature=0.3,
    return_full_text=False      # Mantém apenas a resposta, limpando o prompt
)

Loading weights: 100%|██████████| 254/254 [00:07<00:00, 35.91it/s]
Some parameters are on the meta device because they were offloaded to the disk and cpu.


## 🔄 7. Gerar Triplas com o Modelo Seq2Seq Supreme

Iteramos sobre todos os chunks, montamos o prompt e extraímos a tripla usando um extrator robusto. Apenas chunks com conteúdo significativo são processados.

In [ ]:
def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def is_bad_chunk(text, min_len=80):
    text = text.strip()
    if len(text) < min_len:
        return True
    words = text.split()
    if len(set(words)) < 10:
        return True
    allowed = " .,;:-()[]{}"
    strange_ratio = sum(
        1
        for c in text
        if not c.isalnum() and c not in allowed
    ) / max(len(text), 1)
    return strange_ratio > 0.3


def extract_triple(generated_text, chunk):
    json_match = re.search(r'\{[\s\S]*\}', generated_text)
    
    if not json_match:
        return None
        
    try:
        # Tenta carregar o JSON
        # Tenta carregar o JSON
        data = json.loads(json_match.group(0))

        # Envolvemos o get com str() antes de dar o .strip()
        instruction = str(data.get("Instruction", data.get("instruction", ""))).strip()
        output = str(data.get("Output", data.get("output", ""))).strip()


        if instruction and output:
            return {
                "instruction": instruction,
                "input": chunk.strip(),
                "output": output
            }
    except json.JSONDecodeError:
        # Se não for um JSON válido, ignora a amostra em vez de travar o código
        return None
        
    return None


clean_chunks = [clean_text(c) for c in chunks if not is_bad_chunk(c)]
print(f"Chunks válidos: {len(clean_chunks)}")


def run_model(generator, prompt):
    """Executa qualquer pipeline ou modelo seq2seq direto e retorna o texto gerado."""
    if generator is None:
        return ""

    # Correção para geradores diretos (dicionários de modelo/tokenizer)
    if (
        isinstance(generator, dict)
        and "model" in generator
        and "tokenizer" in generator
    ):
        model = generator["model"]
        tokenizer = generator["tokenizer"]
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True)
        
        # ESSENCIAL: Move os inputs para o mesmo dispositivo (GPU) que o modelo está rodando
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        
        outputs = model.generate(**inputs, max_new_tokens=256)
        return tokenizer.decode(outputs[0], skip_special_tokens=True)

    if not callable(generator):
        return ""

    try:
        # REMOVIDO o max_length=512 para evitar conflito e truncamento do prompt
        output = generator(prompt, truncation=True)
    except TypeError:
        output = generator(prompt)
        
    if isinstance(output, list) and len(output) > 0 and isinstance(output[0], dict):
        return output[0].get('generated_text', '')
        
    return str(output)


Chunks válidos: 3231


In [ ]:
PROMPT_CAUSAL = """Abaixo está uma instrução que descreve uma tarefa, emparelhada com uma entrada que fornece mais contexto. Escreva uma resposta que complete adequadamente o pedido.

### Instruction:
Você é um gerador de datasets para fine-tuning. Com base no contexto técnico fornecido no campo Input, crie exatamente um objeto JSON válido contendo duas chaves textuais:
- "Instruction": Uma pergunta direta, clara e objetiva que um usuário faria sobre os dados do texto.
- "Output": A resposta exata à pergunta formulada, baseando-se estritamente nas informações explícitas do texto, sem inventar nada ou alucinar.
### Input:
{chunk}

### Response:
"""

triplas_causal = []
amostra_chunks = clean_chunks[:120]  # para demonstração, processamos apenas 150 chunks  # para demonstração, processamos apenas 50 chunks

for i, chunk in enumerate(amostra_chunks):
    prompt = PROMPT_CAUSAL.replace("{chunk}", chunk)
    try:
        result = run_model(TunAI_causal, prompt)
        # O modelo causal pode repetir o prompt; removemos a parte inicial se existir
        if result.startswith(prompt):
            generated = result[len(prompt):].strip()
        else:
            generated = result
        triple = extract_triple(generated, chunk)
        if triple is None:
            continue
        if (len(triple["instruction"]) > 15 and len(triple["output"]) > 30):
            triplas_causal.append(triple)
        if i % 10 == 0:
            print(f"Processados {i}/120 chunks (causal)")
    except Exception as e:
        print(f"Erro no chunk {i}: {e}")

print(f"Triplas geradas (causal): {len(triplas_causal)}")

## 🧹 9. Pós‑processamento e Limpeza

Removemos espaços extras, tratamos `input` inválidos e filtramos triplas com output muito curto.

In [ ]:
def clean_triple(triple):
    for key in triple:
        triple[key] = triple[key].strip()
    if triple["input"].lower() in ("none", "null", "n/a", ""):
        triple["input"] = ""
    if len(triple["output"]) < 20 or len(triple["instruction"]) < 5:
        return None
    return triple

def limpar_lista(triplas):
    return [t for t in (clean_triple(x) for x in triplas) if t is not None]

triplas_causal_LLama = limpar_lista(triplas_causal)

print(f"Triplas TunAI após limpeza: {len(triplas_causal_LLama)}")


Triplas GLM após limpeza: 123
Triplas Supreme após limpeza: 150
Triplas TunAI após limpeza: 131
Triplas Apodex após limpeza: 147


## 💾 10. Salvar os Datasets em JSONL

Cada linha será um objeto JSON independente, pronto para uso com `load_dataset`.

In [ ]:
def salvar_jsonl(lista_triplas, nome_arquivo="dataset_gerado.jsonl"):
    """
    Garante o salvamento estrito no formato exigido pelo professor.
    """
    cont_salvos = 0
    with open(nome_arquivo, "w", encoding="utf-8") as f:
        for item in lista_triplas:
            if not item or "instruction" not in item or "output" not in item:
                continue
            dado_final_formatado = {
                "Instruction": item["instruction"],
                "Output": item["output"]
            }
            f.write(json.dumps(dado_final_formatado, ensure_ascii=False) + "\n")
            cont_salvos += 1
    print(f"Sucesso! {cont_salvos} pares tratados e salvos em: {nome_arquivo}")
if 'triplas_causal_TunAI' in globals():
    salvar_jsonl(triplas_causal_LLama, "dataset_causal_TunAI.jsonl")


Sucesso! 123 pares tratados e salvos em: dataset_seq2seq_GLM.jsonl
Sucesso! 150 pares tratados e salvos em: dataset_seq2seq_Supreme.jsonl
Sucesso! 131 pares tratados e salvos em: dataset_causal_TunAI.jsonl
Sucesso! 147 pares tratados e salvos em: dataset_causal_Apodex.jsonl


## 🔬 11. Comparação: Modelo Seq2Seq vs. Causal

Exibimos alguns exemplos gerados por cada modelo para comparar estilos e qualidade.

In [ ]:
def mostrar_exemplo(triplas, titulo, n=3):
    print(f"\n=== {titulo} ===")
    for i, t in enumerate(triplas[:n], 1):
        print(f"\n--- Exemplo {i} ---")
        print(json.dumps(t, indent=2, ensure_ascii=False))

mostrar_exemplo(triplas_causal_LLama, "Causal (TunAI)")


=== Seq2Seq (GLM-ASR-nano) ===

--- Exemplo 1 ---
{
  "instruction": "Explique o conteúdo do texto.",
  "input": "RELATÓRIO DE GESTÃO MENSAGEM DO MINISTRO 2 VISÃO GERAL ORGANIZACIONAL E GOVERNANÇA 3 1.1 Identificação da UPC (Unidade Prestadora de Contas) 4 1.2 Estrutura Organizacional 5 1.3 Cadeia de Valor 9 1.4",
  "output": "RELATRIO DE GESTO MENSAGEM DO MINISTRO 2 VISO GERAL ORGANIZACIONAL E GOVERNANA 3 1.1 Identificaço da UPC (Unidade Prestadora de Contas) 4 1.2 Estrutura Organizacional 5 1.3 Cadeia de Valor 9 1.4"
}

--- Exemplo 2 ---
{
  "instruction": "Explique o conteúdo do texto.",
  "input": "Mapa Estratégico 10 1.5 Políticas Estratégicas 11 1.6 Planejamento e Monitoramento 13 1.7 Descrição dos objetivos do Exercício 17 1.8 Monitoramento dos Instrumentos de Planejamento 19 1.9 Estrutura de",
  "output": "Mapa Estratégico 10 1.5 Polticas Estratégicas 11 1.6 Planejamento e Monitoramento 13 1.7 Descriço dos objetivos do Exerccio 17 1.8 Monitoramento dos Instrumentos de Planejam

## ❓ 12. Exemplo Prático: Pergunta e Resposta com Seq2Seq e Causal

Para ilustrar como os dois paradigmas podem ser usados diretamente como sistemas de QA (Question Answering), vamos tomar um trecho concreto do manual e fazer uma pergunta. O modelo receberá o contexto e a pergunta, e deverá gerar uma resposta.

Escolhemos o primeiro chunk (limpo), que contém a informação da capacidade do refrigerador.

In [ ]:
# Seleciona um chunk de exemplo (o primeiro chunk limpo)
contexto = clean_chunks[0]
pergunta = "Qual é o procedimento recomendado para limpar o filtro de ar do sistema?"

print(f"Contexto:\n{contexto}\n")
print(f"Pergunta: {pergunta}\n")




# ---------- TunAI ----------
prompt_causal_qa = f"""### Instruction:
Responda à pergunta com base no contexto fornecido.

### Input:
Contexto: {contexto}
Pergunta: {pergunta}

### Response:
"""

resposta_causal = run_model(TunAI_causal, prompt_causal_qa)
# Remove possível repetição do prompt
if resposta_causal.startswith(prompt_causal_qa):
    resposta_causal = resposta_causal[len(prompt_causal_qa):].strip()
print(f"\n--- Resposta (Causal - TunAI) ---")
print(resposta_causal.strip())

Contexto:
RELATÓRIO DE GESTÃO MENSAGEM DO MINISTRO 2 VISÃO GERAL ORGANIZACIONAL E GOVERNANÇA 3 1.1 Identificação da UPC (Unidade Prestadora de Contas) 4 1.2 Estrutura Organizacional 5 1.3 Cadeia de Valor 9 1.4

Pergunta: Qual é o procedimento recomendado para limpar o filtro de ar do sistema?

--- Resposta (Seq2Seq - GLM-ASR-nano) ---
1.1 Identificaço da UPC (Unidade Prestadora de Contas) 4 1.2 Estrutura Organizaçional 5 1.3 Cadeia de Valor 9 1.4


[transformers] Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- Resposta (Seq2Seq - Supreme) ---
::::::: RESPONSA  PERGUNTA: RESPONSA: RESPONSA: RESPONSA: RESPONSA: RESPONSA: RESPONSA: RESPONSA: : :         : ::   recomendado  A recomend


[transformers] Both `max_new_tokens` (=256) and `max_length`(=513) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Resposta (Causal - TunAI) ---
Responda à pergunta com base no contexto fornecido.

### Input:
Contexto: GESTÃO ORGANIZACIONAL E GOVERNANÇA 1 1.1 Identificação da UPC (Unidade de Preços e Contas) 2 1.2 Cadeia de Valor 3 2.1 Procedimento para limpar o filtro de ar do sistema 4 2.2 Aplicativo Android 5 2.3 Aplicativo Android 6 3.1
Pergunta: Qual é a melhor opção para limpar o filtro de ar do sistema?

### Response:
Responda à pergunta com base no contexto fornecido.

### Input:
Contexto: GESTÃO ORGANIZACIONAL E GOVERNANÇA 1 1.1 Identificação da UPC (Unidade de Preços e Contas) 2 1.2 Cadeia de Valor 3 2.1 Procedimento para limpar o filtro de ar do sistema 4 2.2 Aplicativo Android

--- Resposta (Causal - Apodex) ---
O filtro de ar do sistema da UPC não está limpo. A UPC é uma instituição financeira, não tem nenhum ativo próprio. O valor da UPC é calculado apenas como valor do seguro e de seus outros ativos. O preço de um seguro é calculado apenas como valor do seguro, o que significa q

## 🎓 13. Conclusão e Próximos Passos

Você agora sabe como transformar documentos em datasets de instruções utilizando **dois paradigmas** de modelos de linguagem. Este dataset pode alimentar o fine‑tuning que aprendemos no notebook anterior, fechando o ciclo completo: **documento → dataset → modelo especializado**.

**Resumo do fluxo:**
1. **Extração** de texto do PDF.
2. **Divisão** em chunks compatíveis com o modelo.
3. **Geração** via LLM (seq2seq ou causal) com prompt estruturado.
4. **Limpeza** e validação do JSON.
5. **Exportação** para JSONL.

**Possíveis melhorias:**
- Usar *few‑shot examples* no prompt para guiar o estilo.
- Aplicar *self‑consistency* (gerar várias respostas e escolher a melhor).
- Validar a fidelidade da resposta em relação ao texto original (usando similaridade de embeddings).
- Substituir os modelos base por versões mais potentes (ex.: Llama 3 8B com 4‑bit quantização).

Agora você está pronto para criar seus próprios dados de treinamento e levar seus modelos ao próximo nível!